## Detect functional groups in MOFs
In this notebook, we will use mofography to detect sepecifc functional groups in MOFs. We will then extend this to look for functional groups in a database of MOFs such as the CoRE MOF database. 
### Algorithm 

Under the hood, mofography uses a graph-based approach to detect functional groups. The algorithm is called subgraph isomorphism and is a well-known problem in computer science. The algorithm is implemented in the [networkx](https://networkx.github.io/) package. Detection occurs based on element and bond types. The source code is in `analyze.py` in the mofography package.

### Steps 

- Load the MOF into ASE
- Load the function group as SMARTS
- Convert both to networkx graphs, via. RDKit
- Detect the functional group in the MOF using subgraph isomorphism
- Make separate atoms object for just selected atoms and bonds in the MOF
- Visualize just hat to confirm


### Functional groups
In this example we are just looking for COO groups, which are extremely common in MOFs. If you want to look for other functional groups, please use SMARTS notation to define the functional group. And some explanation of SMARTS is provided later. 

## Imports
Import the necessary packages.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys
path_to_mgr= Path('c:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography_general/mofography')
sys.path.append(str(path_to_mgr))
import mofography as mgr
import pickle
from ase.io import read
import networkx as nx 
from tqdm import tqdm
import plotly.graph_objects as go

## Grab a dataset
Either grab it from MOFDB or if you have pre saved file read it in to a pandas dataframe.

To grab from MOFDB use this one as a mofography example for setting ranges. This takes a long time, so it's better if you do in once and save the dataframe for future use as a feather or pickle file.  

In [ ]:
df = mgr.get_data_from_mofdb(params={'vf_min': 0.7, 'vf_max': 0.75,'lcd_min':12,'lcd_max':14, 'database':'CoREMOF 2019'})

TO read from saved file, for e.g. `.feather` file, use this example: 

In [ ]:
# path_to_dataset = Path('../../../../my_projects/shape_matching/core_df_gt_05.feather')
# df  = pd.read_feather(path_to_dataset) # put your path here

Let's look at the first 10 MOFs in the dataset.

In [ ]:
df.head(10)

Convert them to ASE atoms objects. DO it once and save it as a pickle file for future use. 

In [ ]:
core_atoms = mgr.atoms_from_mofdb_df(df)
pickle.dump(core_atoms, open('core_atoms.pkl','wb'))

Or if you have it already, load it up here: 

In [ ]:
# core_atoms = pickle.load(open('./core_atoms_list.pkl','rb'))
# core_atoms = pickle.load(open('../../../../my_projects/shape_matching/core_atoms_list.pkl','rb'))

Let's pick one material and test our methodology

In [ ]:
mat_atoms = core_atoms[1]

## How to specify functional moeties? 

Functional moieties, also known as functional groups, can be specified using SMARTS (SMiles ARbitrary Target Specification) notation. SMARTS is a language that allows you to describe molecular patterns. It is used extensively in cheminformatics to search for structural patterns within datasets of molecules.

For example, the SMARTS pattern for a carboxylate group is `[C;X3](-[O;X1])-[O;X1]`.

Here's a breakdown of this pattern:

- `[C;X3]`: This matches a carbon atom (`C`) with three connections (`X3`). The semicolon (`;`) is used to combine different conditions, so `C;X3` matches a carbon atom with exactly three bonds.

- `(-[O;X1])`: This matches an oxygen atom (`O`) with one connection (`X1`), i.e., a single bond. The parentheses are used to group atoms and bonds that should be considered as a unit. The `-` inside the parentheses indicates a single bond.

- `-[O;X1]`: This matches another oxygen atom with a single bond. The `-` outside the parentheses indicates that this oxygen atom is bonded to the previously matched atom (the carbon atom).

So, the SMARTS pattern `[C;X3](-[O;X1])-[O;X1]` matches a carbon atom with three bonds, one of which is to an oxygen atom, and another of which is to another oxygen atom. This is the pattern for a carboxylate group.


In [ ]:
mat_atoms = core_atoms[1]
functional_groups = {
    'carboxylate': '[C;X3](-[O;X1])-[O;X1]',
    # Add more functional groups here
}
matches = mgr.find_functional_groups(mat_atoms, functional_groups)


Let's look at the detected atoms for carboxylate group in the material, if any. 

## Visualize
Now, let's use mofography to visualize and confirm that the correct atoms are detected. 

In [ ]:
# mat_atoms = read(path_to_cif)
mat_atoms = core_atoms[1]
# mat_updated =mgr.update_with_covalent_bonds(mat_atoms)

# Use thse if you need 
# aromatic_molecules = {
#     "Benzene": "c1ccccc1",
#     "Pyridine": "n1ccccc1",
#     "Pyrrole": "c1cc[nH]c1",
#     "Furane": "c1ccoc1",
#     "Thiophene": "c1ccsc1",
#     "Imidazole": "c1nccn1",
#     "Pyrazole": "c1nncn1",
#     "Oxazole": "c1nco1",
#     "Thiazole": "c1ncs1",
#     "Isoxazole": "c1noc1",
#     "Isothiazole": "c1nsc1"
# }
functional_groups = {
    'carboxylate': 'C(=O)=O',
}

matches =mgr.find_functional_groups(mat_atoms,functional_groups)
matching_indices = np.concatenate([list(m.keys()) for m in matches['carboxylate']])
# Create a list of atoms that match the indices
# matching_atoms = []
# for group, indices_list in matches.items():
#     for indices in indices_list:
#         matching_atoms.extend([mat_updated[i] for i in indices.keys()])

frame_subset_trace = mgr.plot_subset_of_atoms(mat_atoms, subset=matching_indices)


In [ ]:
go.Figure(data=[*frame_subset_trace], layout=mgr.epic_layout).show(renderer='notebook_connected', width=800, height=800, scale=2)    

In [ ]:
matches

##  Bonus: Visualize the whole material if you're curious

In [ ]:
frame_trace = mgr.plot_framework_as_mesh(mat_updated,stylize=True)
fig  = go.Figure(data=frame_trace)
fig.update(layout=mgr.epic_layout)
# fig.show(renderer='png', width=800, height=800, scale=2)
fig.show()

In [ ]:
# def find_functional_groups(ase_atoms, functional_groups):
#     from rdkit import Chem
#     from networkx.algorithms import isomorphism
#     import mofography as mgr 

#     # Convert the supercell to a NetworkX graph
#     ase_atoms_updated = mgr.update_with_covalent_bonds(ase_atoms)
#     G = mgr.sgraph_from_atoms(ase_atoms_updated)

#     # Convert functional groups to NetworkX graphs
#     functional_group_graphs = {name: Chem.MolFromSmarts(smarts) for name, smarts in functional_groups.items()}

#     # Custom node match function
#     def node_match(node1, node2):
#         return node1['atomic_number'] == node2['atomic_number']

#     # Custom edge match function
#     def edge_match(edge1, edge2):
#         bond_type1 = edge1['bond_type'].lower()
#         bond_type2 = edge2['bond_type']
#         if bond_type1 == 'aromatic':
#             bond_type1 = 'double'
#         return bond_type1 == bond_type2

#     # Initialize matches with None for all functional groups
#     matches = {name: None for name in functional_groups.keys()}

#     # Search for functional groups in the molecule graph
#     for name, fg_mol in functional_group_graphs.items():
#         fg_G = nx.Graph()
#         for atom in fg_mol.GetAtoms():
#             fg_G.add_node(atom.GetIdx(),
#                           atomic_number=atom.GetAtomicNum(),
#                           formal_charge=atom.GetFormalCharge(),
#                           chirality=atom.GetChiralTag())
#         for bond in fg_mol.GetBonds():
#             bond_type = bond.GetBondType().name.lower()
#             if bond_type == 'aromatic':
#                 bond_type = 'double'
#             fg_G.add_edge(bond.GetBeginAtomIdx(),
#                           bond.GetEndAtomIdx(),
#                           bond_type=bond_type)
#         GM = isomorphism.GraphMatcher(G, fg_G, node_match=node_match, edge_match=edge_match)
#         if GM.subgraph_is_isomorphic():
#             matches[name] = list(GM.subgraph_isomorphisms_iter())

#     return matches

# def find_functional_groups2(ase_atoms, functional_groups):
#     from ase.io import write
#     from rdkit import Chem
#     from rdkit.Chem import AllChem
#     import networkx as nx
#     from networkx.algorithms import isomorphism
#     from io import StringIO

#     # Convert the supercell to an RDKit molecule
#     mol_str = StringIO()
#     write(mol_str, ase_atoms, format='mol')
#     mol = AllChem.MolFromMolBlock(mol_str.getvalue(), removeHs=False)
#     mol.UpdatePropertyCache(strict=False)
#     Chem.GetSymmSSSR(mol)

#     # Convert the RDKit molecule to a NetworkX graph
#     G = nx.Graph()
#     for atom in mol.GetAtoms():
#         G.add_node(atom.GetIdx(),
#                    atomic_num=atom.GetAtomicNum(),
#                    formal_charge=atom.GetFormalCharge(),
#                    chirality=atom.GetChiralTag())
#     for bond in mol.GetBonds():
#         G.add_edge(bond.GetBeginAtomIdx(),
#                    bond.GetEndAtomIdx(),
#                    bond_type=bond.GetBondType())

#     # Convert functional groups to NetworkX graphs
#     functional_group_graphs = {name: Chem.MolFromSmarts(smarts) for name, smarts in functional_groups.items()}

#     # Custom node match function
#     def node_match(node1, node2):
#         return node1['atomic_num'] == node2['atomic_num']

#     # Custom edge match function
#     def edge_match(edge1, edge2):
#         return edge1['bond_type'] == edge2['bond_type']

#     # Search for functional groups in the molecule graph
#     matches = {}
#     for name, fg_mol in functional_group_graphs.items():
#         fg_G = nx.Graph()
#         for atom in fg_mol.GetAtoms():
#             fg_G.add_node(atom.GetIdx(),
#                           atomic_num=atom.GetAtomicNum(),
#                           formal_charge=atom.GetFormalCharge(),
#                           chirality=atom.GetChiralTag())
#         for bond in fg_mol.GetBonds():
#             fg_G.add_edge(bond.GetBeginAtomIdx(),
#                           bond.GetEndAtomIdx(),
#                           bond_type=bond.GetBondType())
#         GM = isomorphism.GraphMatcher(G, fg_G, node_match=node_match, edge_match=edge_match)
#         if GM.subgraph_is_isomorphic():
#             matches[name] = list(GM.subgraph_isomorphisms_iter())

#     return matches

## Apply to CoRE MOFs in High Throughput

Now let's try to apply the functional group detection to a list of coremof atoms, that we have precompiled as shown in the beginning of the notebook. We will look for a few common single ring aromatic groups among the materials in the list. And to speed things up and to make sure that our workflow won't fail even if one or more of the calculations fail, we'll use the 'futures' API from Dask. You can expect another detailed tutorial on using Dask for high throughput materials workflows coming soon. 

It took about 30 minutes for about 7500 coremof materials on a core i7 laptop. Your mileage will vary depending on your hardware and the number of materials you're screening.


(PS: Notice that, if you have multiple functional groups, you can screen for them all at the same time, not one by one.😂)

In [ ]:
functional_groups = {
    'carboxylate': '[C;X3](-[O;X1])-[O;X1]',
    # Add more functional groups here
}
aromatic_molecules = {
    "Benzene": "c1ccccc1",
    "Pyridine": "n1ccccc1",
    "Pyrrole": "c1cc[nH]c1",
    "Furane": "c1ccoc1",
    "Thiophene": "c1ccsc1",
    "Imidazole": "c1nccn1",
    "Pyrazole": "c1nncn1",
    "Oxazole": "c1nco1",
    "Thiazole": "c1ncs1",
    "Isoxazole": "c1noc1",
    "Isothiazole": "c1nsc1"
}
from dask import delayed
from dask.distributed import Client
matches_list = []
for atoms in tqdm(core_atoms):
        matches_list.append(delayed(mgr.find_functional_groups)(atoms,aromatic_molecules))



Start a local dask cluster with 10 processes 

In [ ]:
import dask 
client = Client(n_workers=10)

start running the jobs in the background. 

In [ ]:
futures = dask.persist(*matches_list)

If you look at the 'client' variable, you'll see the dashboard link. You can use that to monitor the progress of the jobs.

In [ ]:
client

I am getting impatient, so let's just run them in the foreground.

In [ ]:
results = client.compute(futures)

Grab all the finished calculations and set the errored ones to None. Each entry in this list is a dictionary of dictionaries. The first dictionary contains all the functional group keys that we are looking for and the cooresponding entry is a nested dictionary that tells the atoms (groups of atom indices) that were flagged for that functional group. Just run it and look at one of the entries to understand better

In [ ]:
outputs = [f.result() if f.status=='finished' else None for f in tqdm(results)]

In [ ]:
outputs[5]

save the matches to a pickle file if you need to for plotting or further analysis.

In [ ]:
pickle.dump(outputs,open('functional_groups_core_gt_05.pkl','wb'))

Let's count the number of materials that contain each of the functional groups

In [ ]:
def count_not_none_entries(list_of_dicts):
    counts = {key: 0 for key in list_of_dicts[0].keys()}
    for dict_ in list_of_dicts:
        for key, value in dict_.items():
            if value is not None:
                counts[key] += 1
    return counts

# Example usage:


print(count_not_none_entries(outputs))

Visualize the materials counts as a bar plot using Plotly

In [ ]:
import plotly.graph_objects as go

def plot_counts(counts):
    fig = go.Figure(data=[
        go.Bar(name='Counts', x=list(counts.keys()), y=list(counts.values()))
    ])
    fig.update_layout(title='Counts of single ring aromatics in CoRE MOF 2019 with vf > 0.5',
                      xaxis_title='single ring aromatics',
                      yaxis_title='Counts')
    fig.write_image('single_ring_aromatics_countes.png', width=800, height=400, scale=2)
    fig.show()

# Example usage:
counts = count_not_none_entries(outputs)
plot_counts(counts)

Pretty cool, right? From here I'll leave it to you to do what ever post processing you want. 